# WordEmbedding 기반 텍스트 분석 연습

1. 적절한 데이터셋을 찾거나 생성하고, 적절한 전처리를 진행한다.
2. Word Embedding model을 이용하여 벡터화
3. 입력한 문자열의 긍/부정을 반환한다 (유사도 활용)

# 1. 02_text_analysis.ipynb 에서 제작한 파일 불러오기
- 맨 마지막에 저장한 파일을 불러옴

In [23]:
import pandas as pd

df = pd.read_csv("C:\\SKN 24_Yunu Jeon\\NLP_practice\\result_for_duolingo_text_anal.csv")
df.head()

,id,rate,comment,comment_tokens_list,comment_tokens,polarity,pos_cos_sim,neu_cos_sim,neg_cos_sim,pred_polarity
0,2dd6b1c2-fcf4-4e99-a743-5fa880d6004a,1,저렴하게 언어공부 가볍게 하기엔 그냥저냥인데 캐릭터들이 참 4가지없네요. 스토리도 ...,"['저렴하다', '언어', '공부', '가볍다', '하다', '그냥저냥', '이다'...",저렴하다 언어 공부 가볍다 하다 그냥저냥 이다 캐릭터 들 참 4 가지 없다 스토리 ...,-1,0.161748,0.209441,0.220928,-1
1,513f5df6-ac02-4a4e-bff2-130c8d470b52,1,최근 버전 업데이트 이후 너무 불편해요. 학습을 제대로 할 수가 없어요. 2년 이상...,"['최근', '버전', '업데이트', '이후', '너무', '불편', '하다', '...",최근 버전 업데이트 이후 너무 불편 하다 학습 제대로 하다 수 없다 2년 이상 하다...,-1,0.274977,0.307943,0.327190,-1
2,7ea1725f-ee3d-4a1e-ad01-132529e41742,1,광고너무뜨네요 원래 안뜨거나 1개였던거 같은데 이제 계속 2개씩 떠서 앱을 사용하는...,"['광고', '너무', '뜨다', '원래', '안', '뜨다', '1', '개', ...",광고 너무 뜨다 원래 안 뜨다 1 개 이다 거 같다 이제 계속 2 개 씩 뜨다 앱 ...,-1,0.132244,0.193762,0.213229,-1
3,4adc04ff-3f84-4b32-ba55-b574d4701151,1,스피킹시 인식을잘못함 하트만날림,"['스피킹', '시', '인식', '잘', '못', '하다', '하트', '날리다']",스피킹 시 인식 잘 못 하다 하트 날리다,-1,0.052863,0.091905,0.106344,-1
4,fd67cb04-2b7a-4d69-9d1c-cf3453fc5fa0,1,도대체 왜 하트를 에너지로 만들고 맞쳐도 1까이고 틀려도 까이는건 공부하지 말라는거...,"['도대체', '왜', '하트', '에너지', '만들다', '맞다', '치다', '...",도대체 왜 하트 에너지 만들다 맞다 치다 1 이다 틀리다 까다 이다 거 공부 하다 ...,-1,0.162678,0.203894,0.237937,-1


# 2. Word Embedding 모델을 활용해 벡터화

### 2-1. 토큰 리스트 변환

In [ ]:
from ast import literal_eval

df['comment_tokens_list'] = df['comment_tokens_list'].apply(literal_eval)

### 2-2. Word2Vec 학습

In [ ]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=preprocessed_sentences,   # 임베딩 모델을 학습하기 위한 코퍼스
    vector_size=100,        # 벡터의 크기: 임베딩 방식의 차원을 결정해주는 내용
    window = 5,             # 앞 뒤로 다섯 개를 쓰겠다
    min_count=5,            # 문장 안에서 min_count보다 적게 등장하면 제거. 다섯 번 이상은 등장해줘야 사용할 수 있음
    sg=0                    # skip-gram: 0 or 1 -> 스킵그램 방식을 적용해서 학슶하겠다. -> 여기선 0이니까 cbow 방식을 활용
)

# CBOW: 가운데 거 가려놓고 중심 단어 예측
# Skit-Gram: 가운데 있는 애만 보고 주변 단어를 예측 -> 주변 단어를 다 볼 수는 없으니까 얼마나 볼 지를 정해야 함 -> 윈도우

In [26]:
from gensim.models import Word2Vec

sentences = df['comment_tokens_list'].tolist()

w2v_model = Word2Vec(
    sentences=sentences,    # 임베딩 모델을 학습할 열
    vector_size=100,        # 벡터의 크기
    window=5,               # 앞 뒤로 5개 쓰기
    min_count=2,            # 적어도 두 번 이상 등장해줘야 쓸 수 있음
    workers=4,              
    sg=1,                   # skip-gram 방식 사용
    seed=42
)

print("어휘 수:", len(w2v_model.wv))

어휘 수: 3240


### 2-3. 특정 단어의 벡터, 유사도, 단어 간 유사도 확인

In [28]:
# 1. 특정 단어 벡터 확인
w2v_model.wv['언어']

array([ 0.42683408,  0.49506894, -0.10782626,  0.23057039, -0.4086455 ,
        0.33537328, -0.327077  , -0.03599946, -0.09560579, -0.03610963,
       -0.6057978 ,  0.2405791 ,  0.18010266,  0.20144232,  0.02414691,
        0.02954824, -0.28954712, -0.2272383 , -0.16153444, -0.03768368,
       -0.34203365, -0.02487535,  0.08424216, -0.09051058,  0.21966806,
        0.3087302 , -0.3475442 , -0.16527458,  0.15461522, -0.7651632 ,
       -0.01572   ,  0.21778353, -0.08647949,  0.35563597,  0.03734816,
       -0.06239007,  0.44781846, -0.05212239,  0.17317426,  0.21812843,
        0.23686858, -0.14994965,  0.0913755 , -0.15922017, -0.07620706,
       -0.2552077 ,  0.00477256,  0.29344535,  0.6610846 , -0.21728726,
       -0.15619975, -0.45245188, -0.19403186,  0.12785417,  0.19003741,
       -0.24543442,  0.12347887, -0.19883968, -0.02674712,  0.0424234 ,
       -0.20359232,  0.01413856, -0.4456858 ,  0.21710098,  0.5867757 ,
       -0.05619161,  0.09257014,  0.00961085, -0.17311661, -0.24

In [30]:
# 유사 단어 보기
w2v_model.wv.most_similar('언어', topn=10)

[('나라', 0.9402229189872742),
 ('외국어', 0.9396646618843079),
 ('러시아어', 0.928450882434845),
 ('기본', 0.9219122529029846),
 ('용', 0.9216060638427734),
 ('지원', 0.9208495616912842),
 ('제외', 0.9200795888900757),
 ('배우다', 0.9184504151344299),
 ('스페인어', 0.9117993712425232),
 ('외', 0.9108626842498779)]

In [31]:
# 2. 단어간 유사도
w2v_model.wv.similarity('언어', '일본어')

0.8510031

# 3. 문장 벡터 만들기
- 문장 전체를 하나의 벡터로 바꾸려면, 각 문장의 단어 벡터 평균 활용
- 리뷰 하나 전체를 벡터로 바꿔야 유사도 기반 감성 판별을 할 수 있음

In [34]:
import numpy as np

def get_sentence_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# 문장벡터 생성
df['sentence_vec'] = df['comment_tokens_list'].apply(
    lambda x: get_sentence_vector(x, w2v_model)
)

df.head()

,id,rate,comment,comment_tokens_list,comment_tokens,polarity,pos_cos_sim,neu_cos_sim,neg_cos_sim,pred_polarity,sentence_vec
0,2dd6b1c2-fcf4-4e99-a743-5fa880d6004a,1,저렴하게 언어공부 가볍게 하기엔 그냥저냥인데 캐릭터들이 참 4가지없네요. 스토리도 ...,"[저렴하다, 언어, 공부, 가볍다, 하다, 그냥저냥, 이다, 캐릭터, 들, 참, 4...",저렴하다 언어 공부 가볍다 하다 그냥저냥 이다 캐릭터 들 참 4 가지 없다 스토리 ...,-1,0.161748,0.209441,0.220928,-1,"[0.19508916, 0.20759875, -0.005307305, -0.0478..."
1,513f5df6-ac02-4a4e-bff2-130c8d470b52,1,최근 버전 업데이트 이후 너무 불편해요. 학습을 제대로 할 수가 없어요. 2년 이상...,"[최근, 버전, 업데이트, 이후, 너무, 불편, 하다, 학습, 제대로, 하다, 수,...",최근 버전 업데이트 이후 너무 불편 하다 학습 제대로 하다 수 없다 2년 이상 하다...,-1,0.274977,0.307943,0.327190,-1,"[0.17309718, 0.26947853, -0.037081663, -0.0075..."
2,7ea1725f-ee3d-4a1e-ad01-132529e41742,1,광고너무뜨네요 원래 안뜨거나 1개였던거 같은데 이제 계속 2개씩 떠서 앱을 사용하는...,"[광고, 너무, 뜨다, 원래, 안, 뜨다, 1, 개, 이다, 거, 같다, 이제, 계...",광고 너무 뜨다 원래 안 뜨다 1 개 이다 거 같다 이제 계속 2 개 씩 뜨다 앱 ...,-1,0.132244,0.193762,0.213229,-1,"[0.2004352, 0.27653998, -0.02484664, -0.070060..."
3,4adc04ff-3f84-4b32-ba55-b574d4701151,1,스피킹시 인식을잘못함 하트만날림,"[스피킹, 시, 인식, 잘, 못, 하다, 하트, 날리다]",스피킹 시 인식 잘 못 하다 하트 날리다,-1,0.052863,0.091905,0.106344,-1,"[0.19082637, 0.3064036, -0.14006566, -0.027380..."
4,fd67cb04-2b7a-4d69-9d1c-cf3453fc5fa0,1,도대체 왜 하트를 에너지로 만들고 맞쳐도 1까이고 틀려도 까이는건 공부하지 말라는거...,"[도대체, 왜, 하트, 에너지, 만들다, 맞다, 치다, 1, 이다, 틀리다, 까다,...",도대체 왜 하트 에너지 만들다 맞다 치다 1 이다 틀리다 까다 이다 거 공부 하다 ...,-1,0.162678,0.203894,0.237937,-1,"[0.23203503, 0.23315968, 0.046950422, -0.08795..."


# 4. 입력한 문자열의 긍/부정 반환

### 4-1. 행 기준으로 감성 반환

In [36]:
def predict_from_sims(row):
    sims = {
        'positive': row['pos_cos_sim'],
        'neutral' : row['neu_cos_sim'],
        'negative': row['neg_cos_sim']
    }
    return max(sims, key=sims.get)

In [ ]:
df['pred_polarity_from_sims'] = df.apply(predict_from_sims, axis=1)

### 4-2. CSV 안의 특정 문장(index) 넣어서 반환

In [38]:
def predict_by_index(i):
    row = df.iloc[i]
    sims = {
        'positive': row['pos_cos_sim'],
        'neutral' : row['neu_cos_sim'],
        'negative': row['neg_cos_sim']
    }
    return max(sims, key=sims.get), sims

In [39]:
predict_by_index(10)

('negative',
 {'positive': 0.1134317635701148,
  'neutral': 0.1652052475786857,
  'negative': 0.1839837018049899})